# AsyncFlow — MM1 Theory vs Simulation (Guided Notebook)

This notebook shows how to:

1. Make imports work inside a notebook (src-layout or package install)
2. Build a **single-server** scenario compatible with **M/M/1** assumptions
3. Run the simulation and collect results
4. Compare theory vs observed KPIs (pretty-printed table)
5. Plot the standard dashboards (latency, throughput, server time series)

> Tip: run this notebook from your project **root folder**.


In [15]:
import sys, importlib

# 1) Svuota tutto ciò che inizia con 'asyncflow' da sys.modules
for m in list(sys.modules):
    if m.startswith("asyncflow"):
        del sys.modules[m]

from asyncflow import AsyncFlow, SimulationRunner
from asyncflow.analysis import MMc, ResultsAnalyzer, SweepAnalyzer
from asyncflow.components import (
    Client, Server, LinkEdge, Endpoint, LoadBalancer, ArrivalsGenerator
)
from asyncflow.settings import SimulationSettings

import simpy

In [16]:
import matplotlib.pyplot as plt
import simpy

# Public AsyncFlow API
from asyncflow import AsyncFlow, SimulationRunner, Sweep
from asyncflow.components import Client, Server, LinkEdge, Endpoint, ArrivalsGenerator
from asyncflow.settings import SimulationSettings
from asyncflow.analysis import MMc, ResultsAnalyzer, SweepAnalyzer
from asyncflow.enums import Distribution

print("Imports OK.")

Imports OK.


## 1) Build an M/M/1-friendly scenario

* **Single server with exponential CPU service**
  One server, one endpoint, exactly **one CPU-bound step** with an **Exponential** service-time RV (mean $E[S]$). No RAM/IO steps in the pipeline.

* **No load balancer**
  Topology has **exactly one server** and **no LB** (no fan-out, no parallelism).

* **Deterministic, very small network latency**
  All edges use a **fixed latency** $\ll 1\,\mathrm{ms}$ so queueing is dominated by CPU service (closer to textbook M/M/1).

* **“Poisson arrivals” via the generator**
 

```mermaid
graph LR;
    rqs1["<b>ArrivalsGenerator</b><br/>id: rqs-1"]
    client1["<b>Client</b><br/>id: client-1"]
    app1["<b>Server</b><br/>id: app-1<br/>Endpoint: /api"]

    rqs1 -- "Edge: gen-client<br/>Latency: 0.0001" --> client1;
    client1 -- "Request<br/>Edge: client-app<br/>Latency: 0.0001" --> app1;
    app1 -- "Response<br/>Edge: app-client<br/>Latency: 0.0001" --> client1;

In [17]:
def build_payload():
    generator = ArrivalsGenerator(
        id="rqs-1",
        lambda_rps=30,
        model=Distribution.POISSON
    )

    client = Client(id="client-1")

    endpoint = Endpoint(
        endpoint_name="/api",
        probability=1.0,
        steps=[
            {
                "kind": "initial_parsing",  # CPU-bound step
                "step_operation": {
                    "cpu_time": {"mean": 0.015, "distribution": "exponential"},
                },
            },
        ],
    )

    server = Server(
        id="app-1",
        server_resources={"cpu_cores": 1, "ram_mb": 2048},
        endpoints=[endpoint],
    )

    e_gen_client = LinkEdge(id="gen-client", source="rqs-1", target="client-1")
    e_client_app = LinkEdge(id="client-app", source="client-1", target="app-1")
    e_app_gen = LinkEdge(id="app-client", source="app-1", target="rqs-1")

    settings = SimulationSettings(
        total_simulation_time=3600,
        sample_period_s=0.05,
    )

    payload = (
        AsyncFlow()
        .add_arrivals_generator(generator)
        .add_client(client)
        .add_servers(server)
        .add_edges(e_gen_client, e_client_app, e_app_gen)
        .add_simulation_settings(settings)
    ).build_payload()
    return payload


## 2) Run the simulation


In [18]:
payload = build_payload()
env = simpy.Environment()
runner = SimulationRunner(env=env, simulation_input=payload)
results: ResultsAnalyzer = runner.run()
print("Done.")

Done.


## 3) MM1 theory vs observed comparison 
If the payload violates MM1 assumptions, a readable error is shown instead.
## Variables (what they represent)

* **λ (lambda)**: mean **arrival rate**, in requests/second.
* **μ (mu)**: mean **service rate**, in requests/second (= 1 / mean service time).
* **ρ (rho)**: **utilization** of the server, ρ = λ / μ (unitless).
* **W**: **mean time in system** (end-to-end latency, queue + service), in seconds.
* **Wq**: **mean waiting time in queue** (before service), in seconds.
* **L**: **mean number in system** (in queue + in service), unitless.
* **Lq**: **mean number in queue**, unitless.
* **E\[S]**: **mean service time** at the server (CPU only), in seconds.


> In the comparison table you’ll see two columns: **Theory** (closed-form M/M/1 values) and **Observed** (estimates from the run). The run is a single execution; “Theory” is the model prediction, “Observed” is what was measured.

---

## How we compute the **Theory** column (M/M/1)

1. **Predicted arrival rate**

$$
\lambda_{\text{Theory}} \;=\; 
\ input data
$$

2. **Predicted service rate** (from the **CPU exponential step** with mean $E[S]$)

$$
\mu_{\text{Theory}} \;=\; \frac{1}{E[S]}
$$

3. **M/M/1 closed forms** (valid when $\lambda_{\text{Theory}} < \mu_{\text{Theory}}$)

$$
\begin{aligned}
\rho_{\text{Theory}} &= \frac{\lambda_{\text{Theory}}}{\mu_{\text{Theory}}} \\
W_{\text{Theory}}    &= \frac{1}{\mu_{\text{Theory}} - \lambda_{\text{Theory}}} \\
W_{q,\text{Theory}}  &= \frac{\rho_{\text{Theory}}}{\mu_{\text{Theory}} - \lambda_{\text{Theory}}} \\
L_{\text{Theory}}    &= \lambda_{\text{Theory}}\, W_{\text{Theory}} \;=\; \frac{\rho_{\text{Theory}}}{1-\rho_{\text{Theory}}} \\
L_{q,\text{Theory}}  &= \lambda_{\text{Theory}}\, W_{q,\text{Theory}} \;=\; \frac{\rho_{\text{Theory}}^{2}}{1-\rho_{\text{Theory}}}
\end{aligned}
$$

If $\lambda_{\text{Theory}} \ge \mu_{\text{Theory}}$, the system is **unstable** and $W, W_q, L, L_q$ **diverge** (we display them as $+\infty$).

---

### How we compute the **Observed** column (from the run, M/M/1)

All estimates are computed **independently** from their own raw measurements (time-series or per-request arrays). We avoid deriving one observed KPI from another, except for explicit fallbacks when a series is unavailable.

1. **Observed arrival rate** (mean throughput over fixed windows)

   $$
   \lambda_{\text{Observed}} \;=\; \text{mean}\big(\text{RPS time series}\big)
   $$

2. **Observed time in system** (client end-to-end latency)

   $$
   W_{\text{Observed}} \;=\; \text{mean}\big(\text{client latencies}\big)
   $$

3. **Observed service rate** (from server service times)

   $$
   \overline{S}=\text{mean}(\text{service\_time}), 
   \quad
   \mu_{\text{Observed}}=
   \begin{cases}
   1/\overline{S} & \overline{S}>0\\[2pt]
   +\infty & \overline{S}=0
   \end{cases}
   $$

4. **Observed waiting time in queue** (from server ready-queue waits)

   $$
   W_{q,\text{Observed}} \;=\; \text{mean}\big(\text{waiting\_time}\big)
   $$

5. **Observed mean number in system** (from time series; **not** via Little’s Law)

   Let $L_{\text{SYSTEM}}[t]$ be the sampled series:

   $$
   L_{\text{Observed}} \;=\; \text{mean}\big(L_{\text{SYSTEM}}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $L_{\text{Observed}} = \lambda_{\text{Observed}} \, W_{\text{Observed}}$.

6. **Observed mean number in queue** (from time series; **not** via Little’s Law)

   In M/M/1 there is a single server; let $L_{q,\text{SERVER}}[t]$ be that server’s queue-length series:

   $$
   L_{q,\text{Observed}} \;=\; \text{mean}\big(L_{q,\text{SERVER}}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $L_{q,\text{Observed}} = \lambda_{\text{Observed}} \, W_{q,\text{Observed}}$.

7. **Observed utilization** (from server utilization series)

   Let $\text{UTIL}[t]\in\{0,1\}$ denote the sampled “server busy” indicator:

   $$
   \rho_{\text{Observed}} \;=\; \text{mean}\big(\text{UTIL}[t]\big)
   $$

   **Fallback (only if the series is unavailable):** $\rho_{\text{Observed}} = \lambda_{\text{Observed}} / \mu_{\text{Observed}}$ (if $\mu_{\text{Observed}}\notin\{0,+\infty\}$, else 0).

> These choices ensure each observed KPI stands on its **own** measurement (time-series or per-request data). Little’s Law based values are used **only** as safe fallbacks when the corresponding series isn’t recorded.


In [19]:
mm1 = MMc()
if mm1.is_compatible(payload):
    mm1.print_comparison(payload, results)  
else:
    print("Payload is not compatible with M/M/1:")
    for reason in mm1.explain_incompatibilities(payload):
        print(" -", reason)


MMc (Random split) — Theory vs Observed
-----------------------------------------------------------------
sym  metric                   theory   observed        abs   rel%
-----------------------------------------------------------------
λ    Arrival rate (1/s)    30.000000  29.942778  -0.057222  -0.19
μ    Service rate (1/s)    66.666667  66.343520  -0.323147  -0.48
rho  Utilization            0.450000   0.451923   0.001923   0.43
L    Mean items in sys      0.818182   0.827317   0.009135   1.12
Lq   Mean items in queue    0.368182   0.375394   0.007212   1.96
W    Mean time in sys (s)   0.027273   0.027570   0.000297   1.09
Wq   Mean waiting (s)       0.012273   0.012497   0.000224   1.83
